In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Reload the dataset
transform = transforms.ToTensor()
train_dataset = torchvision.datasets.MNIST(
    root='../data/',
    train=True,
    download=True,
    transform=transform
)

# DataLoader - feeds data in batches of 128
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Noise function from Part 1
def add_gaussian_noise(image_tensor, noise_factor=0.3):
    noise = torch.randn_like(image_tensor)
    noisy_image = image_tensor + noise_factor * noise
    return torch.clamp(noisy_image, 0., 1.)

print("Setup complete!")
print(f"Total batches per epoch: {len(train_loader)}")

In [ ]:
class DenoisingVAE(nn.Module):
    def __init__(self):
        super(DenoisingVAE, self).__init__()
        # Encoder (Compresses the image)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        
        # Latent Space (The bottleneck)
        self.fc_mu = nn.Linear(64 * 7 * 7, 20)
        self.fc_logvar = nn.Linear(64 * 7 * 7, 20)
        
        # Decoder (Decompresses back to an image)
        self.fc_decode = nn.Linear(20, 64 * 7 * 7)
        self.dec_conv1 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.dec_conv2 = nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1)

    def forward(self, x):
        # 1. Encode
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(-1, 64 * 7 * 7)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        
        # 2. Reparameterize (The magic step)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        
        # 3. Decode
        x = F.relu(self.fc_decode(z))
        x = x.view(-1, 64, 7, 7)
        x = F.relu(self.dec_conv1(x))
        reconstructed = torch.sigmoid(self.dec_conv2(x))
        
        return reconstructed, mu, logvar

print("DenoisingVAE model defined!")

In [ ]:
def vae_loss_function(reconstructed_x, original_x, mu, logvar):
    # Check how closely the output matches the clean image
    BCE = F.binary_cross_entropy(reconstructed_x, original_x, reduction='sum')
    
    # Keep the math organized (KL Divergence)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return BCE + KLD

print("Loss function defined!")

In [ ]:
# Create the model and optimizer
healer_model = DenoisingVAE()
optimizer = optim.Adam(healer_model.parameters(), lr=1e-3)

epochs = 5

print("Starting to train the Healer...")
print("-" * 40)

for epoch in range(epochs):
    total_loss = 0
    
    for batch_idx, (clean_images, _) in enumerate(train_loader):
        
        # 1. Make the images noisy
        noisy_images = add_gaussian_noise(clean_images, noise_factor=0.4)
        
        # 2. Reset the gradients
        optimizer.zero_grad()
        
        # 3. Ask the Healer to fix the noisy images
        reconstructed_images, mu, logvar = healer_model(noisy_images)
        
        # 4. Grade the Healer
        loss = vae_loss_function(reconstructed_images, clean_images, mu, logvar)
        
        # 5. Update the model weights
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/5 complete  |  Loss: {avg_loss:.4f}")

print("-" * 40)
print("Done training!")

In [ ]:
# See how well the Healer actually works!
healer_model.eval()

# Grab 8 test images
test_images = torch.stack([train_dataset[i][0] for i in range(8)])
noisy_test   = add_gaussian_noise(test_images, noise_factor=0.4)

with torch.no_grad():
    healed_images, _, _ = healer_model(noisy_test)

# Plot: Row 1 = Clean, Row 2 = Noisy, Row 3 = Healed
fig, axes = plt.subplots(3, 8, figsize=(16, 6))

for i in range(8):
    axes[0, i].imshow(test_images[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_title('Clean', fontsize=10)

    axes[1, i].imshow(noisy_test[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_title('Noisy', fontsize=10)

    axes[2, i].imshow(healed_images[i].squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0: axes[2, i].set_title('Healed', fontsize=10)

plt.suptitle('Denoising VAE Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()